In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [3]:
df = pd.read_csv("../data/raw/ev_market_2026.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   brand                  2000 non-null   object 
 1   model                  2000 non-null   object 
 2   year                   2000 non-null   int64  
 3   variant                2000 non-null   object 
 4   price_usd              2000 non-null   float64
 5   battery_capacity_kwh   2000 non-null   float64
 6   range_miles            2000 non-null   float64
 7   charging_speed_kw      2000 non-null   float64
 8   acceleration_0_60_mph  2000 non-null   float64
 9   top_speed_mph          2000 non-null   float64
 10  horsepower             2000 non-null   float64
 11  torque_nm              2000 non-null   float64
 12  drive_type             2000 non-null   object 
 13  seating_capacity       2000 non-null   int64  
 14  body_type              2000 non-null   object 
 15  carg

## **Loading and Preprocessing Data**

In [5]:
TARGET = "annual_sales_units"
DROP = ["brand", "model", "variant", "customer_rating"]

df_work = df.drop(columns=DROP)

# Encoding categorical columns to numbers 

cat_cols = ["drive_type", "body_type", "country_of_origin", "market_segment"]
for col in cat_cols:
    df_work[col] = LabelEncoder().fit_transform(df_work[col])

X = df_work.drop(columns=[TARGET])
y = df_work[TARGET]

# Scaling features 

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(X_scaled.shape)


(2000, 19)


## **Fiter Method 1: Pearson Correlation**

In [6]:
TOP_K = 8

# Absolute correlation of each feature with the target 

corr = X_scaled.corrwith(y).abs().sort_values(ascending=False)

print(corr)

charging_speed_kw        0.401654
autopilot_level          0.266799
warranty_years           0.123912
market_segment           0.075325
horsepower               0.067041
torque_nm                0.064073
seating_capacity         0.045239
drive_type               0.030648
price_usd                0.030502
cargo_volume_cubic_ft    0.025823
top_speed_mph            0.023985
acceleration_0_60_mph    0.019917
safety_rating            0.019481
weight_kg                0.013283
range_miles              0.012127
battery_capacity_kwh     0.006791
year                     0.003734
body_type                0.002758
country_of_origin        0.001452
dtype: float64


In [7]:
filter_pearson = corr.head(TOP_K).index.to_list()
print("Pearson top features:", filter_pearson)

Pearson top features: ['charging_speed_kw', 'autopilot_level', 'warranty_years', 'market_segment', 'horsepower', 'torque_nm', 'seating_capacity', 'drive_type']


## **Running baseline models using original and selected features**

In [8]:
# Original data

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LinearRegression()

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)

64122.537795392636
7874376978.586206
88737.68634907159
0.30252304495792404


In [9]:
# After feature selection 

X_selected_features = X_scaled[filter_pearson]

X_train, X_test, y_train, y_test = train_test_split(X_selected_features, y, test_size=0.2, random_state=42)

model_2 = LinearRegression()

model_2.fit(X_train, y_train)
y_pred = model_2.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)

70066.60181429419
8753700234.621204
93561.21116478348
0.22463653929724525


## **Filter Method 2: Mutual Information**

In [10]:
from sklearn.feature_selection import mutual_info_regression

mi_scores = mutual_info_regression(X_scaled,y,random_state=42)

# Wrapping in a series for easy sorting

mi_series = pd.Series(mi_scores, index=X_scaled.columns).sort_values(ascending=False)

print(mi_series)

country_of_origin        0.571714
charging_speed_kw        0.167561
warranty_years           0.143093
autopilot_level          0.139786
market_segment           0.047788
price_usd                0.041691
seating_capacity         0.017403
weight_kg                0.012078
body_type                0.005716
torque_nm                0.005088
drive_type               0.004448
horsepower               0.001738
range_miles              0.000621
cargo_volume_cubic_ft    0.000000
top_speed_mph            0.000000
safety_rating            0.000000
acceleration_0_60_mph    0.000000
battery_capacity_kwh     0.000000
year                     0.000000
dtype: float64


In [11]:
filter_mi = mi_series.head(TOP_K).index.to_list()
print("MI top features", filter_mi)

MI top features ['country_of_origin', 'charging_speed_kw', 'warranty_years', 'autopilot_level', 'market_segment', 'price_usd', 'seating_capacity', 'weight_kg']


In [12]:
X_selected_features = X_scaled[filter_mi]

X_train, X_test, y_train, y_test = train_test_split(X_selected_features, y, test_size=0.2, random_state=42)

model_3 = LinearRegression()

model_3.fit(X_train, y_train)
y_pred = model_3.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)

64833.339793637395
8122697794.872266
90126.01064549715
0.28052790206753886


## **Filter Method 3: ANOVA F-score**

In [15]:
from sklearn.feature_selection import f_regression

f_scores, p_values = f_regression(X_scaled, y)

f_series = pd.Series(f_scores, index=X_scaled.columns).sort_values(ascending=False)

print(f_series)


charging_speed_kw        384.331215
autopilot_level          153.120937
warranty_years            31.155867
market_segment            11.401129
horsepower                 9.020521
torque_nm                  8.236316
seating_capacity           4.097407
drive_type                 1.878488
price_usd                  1.860569
cargo_volume_cubic_ft      1.333166
top_speed_mph              1.150064
acceleration_0_60_mph      0.792862
safety_rating              0.758541
weight_kg                  0.352604
range_miles                0.293874
battery_capacity_kwh       0.092135
year                       0.027861
body_type                  0.015195
country_of_origin          0.004215
dtype: float64


In [16]:
filter_anova = f_series.head(TOP_K).index.tolist()
print("F-score top features:", filter_anova)

F-score top features: ['charging_speed_kw', 'autopilot_level', 'warranty_years', 'market_segment', 'horsepower', 'torque_nm', 'seating_capacity', 'drive_type']


## **Wrapper Method: 1 RFE (Recursive Feature Elimination)**

In [21]:
from sklearn.ensemble import RandomForestRegressor

TOP_K = 8

base_estimator = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

from sklearn.feature_selection import RFE

rfe = RFE(
    estimator=base_estimator,
    n_features_to_select=TOP_K,
    step=1
)

rfe.fit(X_scaled,y)

wrapper_rfe = [f for f, selected in zip(X_scaled.columns, rfe.support_) if selected]

rfe_ranking = dict(zip(X_scaled.columns, rfe.ranking_))

print("RFE selected:", wrapper_rfe)
print("\nFull ranking (1 = best):")
print(pd.Series(rfe_ranking).sort_values())




RFE selected: ['price_usd', 'charging_speed_kw', 'acceleration_0_60_mph', 'cargo_volume_cubic_ft', 'weight_kg', 'autopilot_level', 'country_of_origin', 'warranty_years']

Full ranking (1 = best):
warranty_years            1
price_usd                 1
charging_speed_kw         1
acceleration_0_60_mph     1
country_of_origin         1
autopilot_level           1
weight_kg                 1
cargo_volume_cubic_ft     1
torque_nm                 2
range_miles               3
battery_capacity_kwh      4
top_speed_mph             5
horsepower                6
year                      7
body_type                 8
drive_type                9
safety_rating            10
seating_capacity         11
market_segment           12
dtype: int32
